# Cooperation for compounds from other members: consumption overlap

In [83]:
import pandas as pd
import json
import seaborn as sns
import numpy as np

import networkx as nx

from pyvis.network import Network

import sys

sys.path.append("../functions/")
import networks

In [84]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [85]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = media_db[media_db.medium=="M9"].compound.values
compounds = list(compounds)
compounds.remove("glc__D")
m9_compounds = ["M_"+met+"__91__e__93__" for met in compounds]

In [86]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]
results = results[~results.compound.isin(m9_compounds)]

In [87]:
results[results.donor=="environment"]

,donor,receiver,compound,mass_rate,rate,frequency,community,medium,mass_rate*frequency
4,environment,Bacteroides_uniformis_ATCC_8492,M_malt__91__e__93__,5.220897e-02,0.152526,0.985,NONE,LB_extend,5.142583e-02
6,environment,Faecalibacterium_prausnitzii_A2_165,M_ribflv__91__e__93__,4.384658e-02,0.116501,1.000,NONE,LB_extend,4.384658e-02
7,environment,Bacteroides_uniformis_ATCC_8492,M_fum__91__e__93__,4.099854e-02,0.359459,0.781,NONE,LB_extend,3.201986e-02
9,environment,Bacteroides_uniformis_ATCC_8492,M_cellb__91__e__93__,3.357823e-02,0.098097,0.020,NONE,LB_extend,6.715646e-04
14,environment,Faecalibacterium_prausnitzii_A2_165,M_malt__91__e__93__,1.840314e-02,0.053764,0.774,NONE,LB_extend,1.424403e-02
...,...,...,...,...,...,...,...,...,...
2016,environment,Desulfovibrio_piger_ATCC_29098,M_pnto_R__91__e__93__,2.569624e-07,0.000001,1.000,NONE,LB_extend,2.569624e-07
2019,environment,Bacteroides_vulgatus_ATCC_8482,M_spmd__91__e__93__,2.550410e-07,0.000002,1.000,NONE,LB_extend,2.550410e-07
2030,environment,Eubacterium_rectale_ATCC_33656,M_ptrc__91__e__93__,2.417336e-07,0.000003,0.645,NONE,LB_extend,1.559182e-07
2056,environment,Eggerthella_lenta_DSM_2243,M_ptrc__91__e__93__,1.861179e-07,0.000002,1.000,NONE,LB_extend,1.861179e-07


## Cooperation based on COUNT (Only metabolites from metabolomics + freq_cutoff = 0.1 !!)

In [93]:

network_data = networks.cooperation_network_data(results,metabolomics_mets_only=True,use_count=True,freq_cutoff=0.1)

In [94]:
float(network_data.min())

1.0

In [95]:
float(network_data.max())

3.0

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Closeness centrality**: The reciprocal of sum of shortest paths to i (Incoming path for directed graph)
- **Degree centrality**: The fraction of nodes that node i is connected to. (Sum of incoming and outgoing edges for directed graphs)
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes (right eigenvector centrality is calculated)


In [96]:
centrality_metrics_SP = networks.cooperation_centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Bacteroides_ovatus_ATCC_8483,0.000000,0.000000
Faecalibacterium_prausnitzii_A2_165,0.033333,0.263158
Bacteroides_vulgatus_ATCC_8482,0.022222,0.277778
Eubacterium_rectale_ATCC_33656,0.042593,0.294118
Collinsella_aerofaciens_ATCC_25986,0.025926,0.327273
Eggerthella_lenta_DSM_2243,0.000000,0.365714
Blautia_hydrogenotrophica_DSM_10507,0.040741,0.378947
Bacteroides_uniformis_ATCC_8492,0.077778,0.384615
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.385714
Desulfovibrio_piger_ATCC_29098,0.000000,0.400000


In [97]:
centrality_metrics_notSP = networks.cooperation_centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")
centrality_metrics_notSP

,degree (data),eigenvector (data)
Collinsella_aerofaciens_ATCC_25986,2,1.908094e-01
Eggerthella_lenta_DSM_2243,2,3.151447e-01
Desulfovibrio_piger_ATCC_29098,4,5.552051e-01
Bacteroides_vulgatus_ATCC_8482,4,1.362306e-01
Blautia_hydrogenotrophica_DSM_10507,4,4.632703e-01
Bacteroides_thetaiotaomicron_VPI_5482,5,1.575738e-01
Bacteroides_uniformis_ATCC_8492,5,1.575736e-01
Faecalibacterium_prausnitzii_A2_165,7,1.908094e-01
Eubacterium_rectale_ATCC_33656,8,2.827441e-01
Bacteroides_ovatus_ATCC_8483,8,8.595319e-09


In [98]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"]

In [99]:
net = networks.cooperation_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("network_file.html")

network_file.html



## Cooperation based on COUNT (All metabolites + freq_cutoff=0.1!!)

In [100]:

network_data = networks.cooperation_network_data(results,metabolomics_mets_only=False,use_count=True,freq_cutoff=0.1)

In [101]:
float(network_data.min())

1.0

In [102]:
float(network_data.max())

4.0

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Closeness centrality**: The reciprocal of sum of shortest paths to i (Incoming path for directed graph)
- **Degree centrality**: The fraction of nodes that node i is connected to. (Sum of incoming and outgoing edges for directed graphs)
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes (right eigenvector centrality is calculated)


In [103]:
centrality_metrics_SP = networks.cooperation_centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Bacteroides_uniformis_ATCC_8492,0.016667,0.648000
Bacteroides_ovatus_ATCC_8483,0.000000,0.648000
Bacteroides_vulgatus_ATCC_8482,0.094444,0.943689
Blautia_hydrogenotrophica_DSM_10507,0.000000,0.981818
Eubacterium_rectale_ATCC_33656,0.020370,1.002062
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,1.012500
Faecalibacterium_prausnitzii_A2_165,0.233333,1.104545
Eggerthella_lenta_DSM_2243,0.005556,1.313514
Clostridium_hiranonis_TO_931_DSM_13275,0.220370,1.331507
Collinsella_aerofaciens_ATCC_25986,0.050000,1.408696


In [104]:
centrality_metrics_notSP = networks.cooperation_centrality_metrics(network_data,SP_centrality=False).sort_values("degree (data)")
centrality_metrics_notSP

,degree (data),eigenvector (data)
Blautia_hydrogenotrophica_DSM_10507,13,0.201871
Eggerthella_lenta_DSM_2243,15,0.269947
Bacteroides_thetaiotaomicron_VPI_5482,20,0.234536
Eubacterium_rectale_ATCC_33656,21,0.234213
Desulfovibrio_piger_ATCC_29098,23,0.573821
Bacteroides_vulgatus_ATCC_8482,23,0.266131
Collinsella_aerofaciens_ATCC_25986,23,0.383350
Bacteroides_uniformis_ATCC_8492,25,0.113150
Bacteroides_ovatus_ATCC_8483,27,0.118280
Clostridium_hiranonis_TO_931_DSM_13275,31,0.325354


In [105]:
closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"]

In [106]:
net = networks.cooperation_network(network_data,closeness_centrality_met=closeness_centrality_met)
net.show("network_file.html")

network_file.html


## Cooperation based on FREQUENCY (metabolomics_mets_only=True,freq_cutoff=0.1)

In [107]:

network_data = networks.cooperation_network_data(results,metabolomics_mets_only=True,use_count=False,freq_cutoff=0.1)

In [108]:
float(network_data.min())

0.104

In [109]:
float(network_data.max())

1.0

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Closeness centrality**: The reciprocal of sum of shortest paths to i (Incoming path for directed graph)
- **Degree centrality**: The fraction of nodes that node i is connected to. (Sum of incoming and outgoing edges for directed graphs)
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes (right eigenvector centrality is calculated)


In [110]:
centrality_metrics_SP = networks.cooperation_centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Bacteroides_ovatus_ATCC_8483,0.000000,0.000000
Bacteroides_vulgatus_ATCC_8482,0.022222,0.036686
Eubacterium_rectale_ATCC_33656,0.033333,0.041554
Faecalibacterium_prausnitzii_A2_165,0.033333,0.045196
Collinsella_aerofaciens_ATCC_25986,0.027778,0.057292
Desulfovibrio_piger_ATCC_29098,0.000000,0.063748
Blautia_hydrogenotrophica_DSM_10507,0.038889,0.064266
Clostridium_hiranonis_TO_931_DSM_13275,0.288889,0.087565
Eggerthella_lenta_DSM_2243,0.000000,0.099973
Bacteroides_thetaiotaomicron_VPI_5482,0.000000,0.104454


In [111]:

centrality_metrics_notSP = networks.cooperation_centrality_metrics(network_data,SP_centrality=False,degree_type="in").sort_values("degree (data)")
centrality_metrics_notSP

,degree (data),eigenvector (data)
Bacteroides_ovatus_ATCC_8483,0.000000,1.603952e-08
Collinsella_aerofaciens_ATCC_25986,0.175000,1.295225e-01
Faecalibacterium_prausnitzii_A2_165,0.175000,1.295225e-01
Eubacterium_rectale_ATCC_33656,0.229000,1.250546e-01
Blautia_hydrogenotrophica_DSM_10507,0.316000,1.721150e-01
Desulfovibrio_piger_ATCC_29098,0.413500,2.027116e-01
Bacteroides_vulgatus_ATCC_8482,0.831000,3.655149e-02
Clostridium_hiranonis_TO_931_DSM_13275,1.653500,3.063978e-01
Eggerthella_lenta_DSM_2243,2.000000,7.286143e-01
Bacteroides_uniformis_ATCC_8492,2.237000,3.217339e-01


In [112]:
closeness_centrality_met = centrality_metrics_notSP["degree (data)"]

In [113]:
net = networks.cooperation_network(network_data,closeness_centrality_met=closeness_centrality_met,use_count=False)
net.show("network_file.html")

network_file.html


In [114]:

net = networks.cooperation_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=0.1,use_count=False)
net.show("network_file.html")

network_file.html


## Cooperation based on FREQUENCY (metabolomics_mets_only=False,freq_cutoff=0.1)

In [190]:

network_data = networks.cooperation_network_data(results,metabolomics_mets_only=False,use_count=False,freq_cutoff=0.1)

In [191]:
float(network_data.min())

0.108

In [192]:
float(network_data.max())

0.967

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Closeness centrality**: The reciprocal of sum of shortest paths to i (Incoming path for directed graph)
- **Degree centrality**: The fraction of nodes that node i is connected to. (Sum of incoming and outgoing edges for directed graphs)
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes (right eigenvector centrality is calculated)


In [193]:
centrality_metrics_SP = networks.cooperation_centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Faecalibacterium_prausnitzii_A2_165,0.044444,0.117248
Bacteroides_uniformis_ATCC_8492,0.066667,0.145729
Bacteroides_ovatus_ATCC_8483,0.111111,0.145917
Bacteroides_vulgatus_ATCC_8482,0.022222,0.158850
Blautia_hydrogenotrophica_DSM_10507,0.011111,0.162281
Collinsella_aerofaciens_ATCC_25986,0.033333,0.183285
Bacteroides_thetaiotaomicron_VPI_5482,0.033333,0.185868
Desulfovibrio_piger_ATCC_29098,0.000000,0.196743
Eubacterium_rectale_ATCC_33656,0.011111,0.210130
Clostridium_hiranonis_TO_931_DSM_13275,0.188889,0.223515


In [194]:

centrality_metrics_notSP = networks.cooperation_centrality_metrics(network_data,SP_centrality=False,degree_type="in").sort_values("degree (data)")
centrality_metrics_notSP

,degree (data),eigenvector (data)
Faecalibacterium_prausnitzii_A2_165,1.003000,0.169900
Blautia_hydrogenotrophica_DSM_10507,1.096000,0.178371
Bacteroides_ovatus_ATCC_8483,1.473000,0.198220
Bacteroides_vulgatus_ATCC_8482,1.833083,0.274862
Collinsella_aerofaciens_ATCC_25986,1.878333,0.314651
Eubacterium_rectale_ATCC_33656,1.881000,0.338701
Bacteroides_uniformis_ATCC_8492,1.890000,0.225705
Desulfovibrio_piger_ATCC_29098,2.074417,0.328946
Eggerthella_lenta_DSM_2243,2.214833,0.339855
Clostridium_hiranonis_TO_931_DSM_13275,2.360500,0.360876


In [195]:
closeness_centrality_met = centrality_metrics_notSP["degree (data)"]

In [196]:
net = networks.cooperation_network(network_data,closeness_centrality_met=closeness_centrality_met,use_count=False)
net.show("network_file.html")

network_file.html


## Cooperation based on MASS_RATE x FREQUENCY (metabolomics_mets_only=False,freq_cutoff=0.1)

In [197]:

network_data = networks.cooperation_network_data(results,metric="mass_rate*frequency",metabolomics_mets_only=False,use_count=False,freq_cutoff=0.1)

In [198]:
network_data = network_data*1000

In [199]:
float(network_data.min())

3.0342453517523986e-05

In [200]:
float(network_data.max())

3.850323721306155

**Network topological properties**


- **Betweeness_centrality**: Sum of shortest paths that pass through node i
- **Closeness centrality**: The reciprocal of sum of shortest paths to i (Incoming path for directed graph)
- **Degree centrality**: The fraction of nodes that node i is connected to. (Sum of incoming and outgoing edges for directed graphs)
- **Eigenvector centrality**: Higher if node i is connected to other well connected notes (right eigenvector centrality is calculated)


In [201]:
centrality_metrics_SP = networks.cooperation_centrality_metrics(network_data).sort_values("closeness (1/data)")
centrality_metrics_SP

,betweeness (1/data),closeness (1/data)
Desulfovibrio_piger_ATCC_29098,0.000000,0.005547
Bacteroides_uniformis_ATCC_8492,0.022222,0.010236
Faecalibacterium_prausnitzii_A2_165,0.211111,0.010577
Bacteroides_ovatus_ATCC_8483,0.011111,0.011309
Eubacterium_rectale_ATCC_33656,0.000000,0.011586
Bacteroides_vulgatus_ATCC_8482,0.277778,0.011706
Collinsella_aerofaciens_ATCC_25986,0.066667,0.012709
Eggerthella_lenta_DSM_2243,0.100000,0.013349
Clostridium_hiranonis_TO_931_DSM_13275,0.488889,0.013546
Bacteroides_thetaiotaomicron_VPI_5482,0.433333,0.013547


In [202]:

centrality_metrics_notSP = networks.cooperation_centrality_metrics(network_data,SP_centrality=False,degree_type="in").sort_values("degree (data)")
centrality_metrics_notSP

,degree (data),eigenvector (data)
Desulfovibrio_piger_ATCC_29098,0.033806,0.005098
Bacteroides_uniformis_ATCC_8492,0.160988,0.003032
Faecalibacterium_prausnitzii_A2_165,0.202924,0.017606
Eubacterium_rectale_ATCC_33656,0.216511,0.029258
Blautia_hydrogenotrophica_DSM_10507,0.379166,0.015140
Bacteroides_vulgatus_ATCC_8482,0.608653,0.030321
Collinsella_aerofaciens_ATCC_25986,0.699953,0.129691
Eggerthella_lenta_DSM_2243,0.837560,0.312002
Clostridium_hiranonis_TO_931_DSM_13275,2.873665,0.703279
Bacteroides_thetaiotaomicron_VPI_5482,6.487837,0.621413


In [203]:
closeness_centrality_met = centrality_metrics_notSP["degree (data)"]

In [204]:
net = networks.cooperation_network(network_data,closeness_centrality_met=closeness_centrality_met,use_count=False)
net.show("network_file.html")

network_file.html


In [205]:

net = networks.cooperation_network(network_data,closeness_centrality_met=closeness_centrality_met,cutoff=0.1,use_count=False)
net.show("network_file.html")

network_file.html
